# R2-04 · Fundamentos de Machine Learning — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 10 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Plantear un problema **supervisado**: features (`X`) y objetivo (`y`).
2. Fabricar una *feature* de **rezago** (`shift`).
3. Separar entrenamiento y prueba con **corte temporal**.
4. Medir un **baseline** ingenuo y entrenar una **regresión lineal** (`fit`/`predict`).
5. Evaluar con **MAE** en prueba y **predecir** un caso nuevo.

**Competencia de salida:** entrenar y evaluar con honestidad un primer modelo de regresión que pronostica el PM2.5.

**Dato real:** PM2.5 diario, temperatura y humedad de Cerro Navia 2020–2025 (SINCA, MMA).

**Entregable:** que las **5 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "aire_diario.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)

df = pd.read_csv(CSV, parse_dates=["fecha"])
# Estación ancla: Cerro Navia (RM), ordenada por fecha (un día por fila, sin días faltantes).
cn = df[df["estacion"] == "Cerro Navia"].sort_values("fecha").reset_index(drop=True)
print("Cerro Navia:", cn.shape, "| desde", cn["fecha"].min().date(), "hasta", cn["fecha"].max().date())
cn.head()

## ¿Qué es el *machine learning*?

Programar «a la antigua» es **escribir reglas a mano**: *si hace frío y no hay viento, el aire
estará malo*. El *machine learning* lo da vuelta: le mostramos al computador **muchos ejemplos**
con la respuesta conocida y dejamos que **descubra la regla solo**.

> 🧠 **Analogía.** A nadie le enseñaron a reconocer un día de smog con una lista de reglas: viste
> muchos inviernos y aprendiste el patrón. Un modelo hace eso con datos.

### Aprendizaje supervisado
- **Features (`X`):** lo que el modelo usa para predecir — temperatura, humedad, mes, día de la semana y el PM2.5 de ayer.
- **Objetivo (`y`):** lo que queremos predecir — el **PM2.5 del día** en Cerro Navia.

| Tipo | Predice | Ejemplo |
| --- | --- | --- |
| **Regresión** | un **número** | µg/m³ de PM2.5 mañana |
| **Clasificación** | una **categoría** | ¿mañana se supera la norma de 50, sí o no? |

Hoy hacemos **regresión**. La pregunta pública es concreta: si pudieras anticipar el PM2.5 de
mañana, podrías **avisar antes** a la población y a los servicios de salud.

## 1. Fabricar las *features* (incluido el rezago)

Un modelo solo puede usar lo que le damos en columnas. La *feature* más valiosa para pronosticar es, casi siempre, **el valor de ayer** (el *rezago* o *lag*): `shift(1)` corre la columna una fila hacia abajo. Como `cn` está ordenada por fecha y no le faltan días, la fila anterior es exactamente ayer.

### ✍️ Ejercicio 1 — Crea `pm25_ayer`, `mes` y `dia_semana`

In [ ]:
cn["pm25_ayer"] = cn["pm25"].shift(1)
cn["mes"] = cn["fecha"].dt.month
cn["dia_semana"] = cn["fecha"].dt.dayofweek          # 0 = lunes ... 6 = domingo
FEATURES = ["temp_media", "humedad_media", "mes", "dia_semana", "pm25_ayer"]
datos = cn.dropna(subset=FEATURES + ["pm25"])         # solo días completos
print("días con todas las features:", len(datos))
datos[["fecha", "pm25", "pm25_ayer", "mes", "dia_semana"]].head()

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert np.allclose(datos["pm25_ayer"].astype(float), cn["pm25"].shift(1).loc[datos.index])
    assert datos["dia_semana"].between(0, 6).all()
    assert datos[FEATURES].notna().all().all()
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'cn["pm25"].shift(1) trae el valor de la fila anterior; cn["fecha"].dt.dayofweek da 0..6.')
    print("   Detalle:", e)

## 2. La regla de oro: probar con datos que el modelo no vio — y del **futuro**

Si evalúas con los mismos datos con que entrenaste, el modelo puede «memorizar» y parecer perfecto. Por eso se reserva un **conjunto de prueba**. Pero aquí hay un detalle clave: queremos **pronosticar el futuro**, así que la prueba debe imitar ese uso. **No barajamos** los días al azar: entrenamos con **2020–2024** y probamos en **2025**, un año completo que el modelo nunca vio.

> ⚠️ Barajar una serie de tiempo mete días de 2025 en el entrenamiento y deja que el modelo «vea» vecinos del futuro: la evaluación ya no mide lo que importa (y puede quedar maquillada). En R2-07 profundizarás la validación temporal.

### ✍️ Ejercicio 2 — Corte temporal: entrenamiento antes de 2025, prueba en 2025

In [ ]:
corte = pd.Timestamp("2025-01-01")
train = datos[datos["fecha"] < corte]
test = datos[datos["fecha"] >= corte]
X_train, y_train = train[FEATURES], train["pm25"]
X_test, y_test = test[FEATURES], test["pm25"]
print("entrenamiento:", len(train), "días (2020–2024) | prueba:", len(test), "días (2025)")

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert train["fecha"].max() < test["fecha"].min(), "Todo el entrenamiento debe ser anterior a la prueba"
    assert len(train) + len(test) == len(datos)
    assert (test["fecha"].dt.year == 2025).all()
    assert list(X_train.columns) == FEATURES
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'train = datos[datos["fecha"] < corte] y test = datos[datos["fecha"] >= corte].')
    print("   Detalle:", e)

## 3. Un *baseline*: la vara que el modelo debe superar

Antes de entrenar nada, define una predicción **ingenua**. Para el aire, la más dura de batir es la **persistencia**: «mañana será igual que hoy» (predecir con `pm25_ayer`). Si tu modelo no le gana, no sirve — por sofisticado que sea. La métrica: **MAE** (error absoluto medio), en µg/m³.

### ✍️ Ejercicio 3 — MAE del baseline de persistencia (y de «predecir el promedio»)

In [ ]:
pred_base = X_test["pm25_ayer"]
mae_base = mean_absolute_error(y_test, pred_base)
mae_media = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
print("MAE «mañana = hoy»:", round(mae_base, 2), "µg/m³ | MAE «siempre el promedio»:", round(mae_media, 2))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert abs(mae_base - mean_absolute_error(y_test, X_test["pm25_ayer"])) < 1e-9
    assert mae_base < mae_media, "La persistencia es mucho mejor que predecir siempre el promedio"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'pred_base = X_test["pm25_ayer"] y mae_base = mean_absolute_error(y_test, pred_base).')
    print("   Detalle:", e)

## 4. Entrenar tu primer modelo: `fit` y `predict`

La **regresión lineal** busca la mejor combinación «número × feature» para acercarse a `y`. En scikit-learn todos los modelos se usan igual:

```python
modelo = LinearRegression()
modelo.fit(X_train, y_train)    # 1) APRENDER del pasado
y_pred = modelo.predict(X_test)  # 2) PREDECIR días que no vio
```

Compara el MAE en prueba con el del baseline, y con el MAE en **entrenamiento**: si el de entrenamiento fuera mucho menor que el de prueba, sería la alarma de **sobreajuste**.

### ✍️ Ejercicio 4 — Entrena, predice 2025 y compara con el baseline

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
mae_lr = mean_absolute_error(y_test, y_pred)
mae_lr_train = mean_absolute_error(y_train, modelo.predict(X_train))
print("MAE prueba:", round(mae_lr, 2), "| MAE entrenamiento:", round(mae_lr_train, 2), "| baseline:", round(mae_base, 2))
print("Coeficientes (µg/m³ por unidad de cada feature):")
print(pd.Series(modelo.coef_, index=FEATURES).round(2))

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert hasattr(modelo, "coef_"), "Falta modelo.fit(X_train, y_train)"
    assert abs(mae_lr - mean_absolute_error(y_test, modelo.predict(X_test))) < 1e-9
    assert mae_lr < mae_base, "El modelo debería ganarle (por poco) al baseline"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'modelo = LinearRegression(); modelo.fit(X_train, y_train); y_pred = modelo.predict(X_test).')
    print("   Detalle:", e)

In [ ]:
# (ilustración) 2025: lo real vs lo que predijo el modelo
try:
    plt.figure(figsize=(10, 3.8))
    plt.plot(test["fecha"], y_test, color="#999999", lw=1, label="real")
    plt.plot(test["fecha"], y_pred, color="#4240e5", lw=1.2, label="modelo lineal")
    plt.axhline(50, color="#c0392b", ls="--", label="norma 24 h (50 µg/m³)")
    plt.ylabel("PM2.5 (µg/m³)"); plt.title("Cerro Navia 2025 — datos que el modelo nunca vio")
    plt.legend(); plt.tight_layout(); plt.show()
except (NameError, TypeError, ValueError, AttributeError):
    print("ℹ️ Completa el ejercicio 4 para ver este gráfico.")

## 5. Usar el modelo: un día nuevo

Para esto entrenamos modelos: **estimar lo que todavía no ocurre**. scikit-learn pide la entrada con la **misma forma** que el entrenamiento: una tabla con las mismas columnas, en el mismo orden.

### ✍️ Ejercicio 5 — Pronostica un miércoles de julio frío y húmedo

In [ ]:
dia_nuevo = pd.DataFrame([{"temp_media": 7.0, "humedad_media": 85.0, "mes": 7,
                           "dia_semana": 2, "pm25_ayer": 60.0}])[FEATURES]
pm25_estimado = float(modelo.predict(dia_nuevo)[0])
supera_norma = pm25_estimado > 50
print("PM2.5 estimado:", round(pm25_estimado, 1), "µg/m³ | ¿sobre la norma?", supera_norma)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert abs(pm25_estimado - float(modelo.predict(dia_nuevo)[0])) < 1e-6
    assert supera_norma == (pm25_estimado > 50)
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'float(modelo.predict(dia_nuevo)[0]) y supera_norma = pm25_estimado > 50.')
    print("   Detalle:", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)

1. El modelo usa la **temperatura y la humedad del mismo día** que predice. En la vida real, para
   pronosticar mañana necesitarías el **pronóstico meteorológico** de mañana. ¿Qué le pasaría al
   error si el pronóstico del tiempo también se equivoca?
2. La regresión lineal lee `mes` como un número: para ella, diciembre (12) es «más» que julio (7).
   ¿Tiene sentido? (En R2-05 verás modelos que no tienen ese problema.)

## Errores típicos

- **Entrenar con los datos de prueba** → `fit` siempre con `X_train`, nunca con `X_test`.
- **Barajar una serie de tiempo** → el modelo «ve el futuro» y la evaluación se infla.
- **No tener baseline** → un MAE de 7 µg/m³ no dice nada si no sabes que «mañana = hoy» ya logra ~7,4.
- **`y` con doble corchete** → el objetivo va con uno (`train["pm25"]`), las features con lista.

## Cierre

- **Supervisado:** features (`X`) → objetivo (`y`); hoy, regresión del PM2.5.
- **Corte temporal:** entrenar con el pasado, probar con el futuro.
- **Baseline primero:** el modelo lineal le gana a la persistencia, pero por poco — y eso es información honesta.
- **`fit` / `predict`:** el patrón es el mismo para todos los modelos que vienen.

> *En R2-05 cambiarás la recta por árboles y bosques — mismo dato, mismo corte, misma vara.*